In [1]:
%load_ext autoreload
%autoreload 2

# VARSTok Audio Tokenization Example

This notebook demonstrates the VARSTok audio tokenizer - a SOTA discrete codec model having variable frame rate achieved through clustering.

**Key Properties:**
- Input sample rate: 24 kHz
- Output sample rate: 24 kHz
- Codebook size: 4096 tokens
- Token rate: <40 Hz 
- Architecture: Encoder -> Temporal-Aware Clustering -> Mean pooling (Vectors) -> VQ  with single codebook -> Codes -> Use extended indexing to get tokens -> Decoder
- Supports speech, audio, and music


## Setup and Imports


In [2]:
import sys
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


/users/arsaikia/benchmark-audio-tokenizer/.venv-varstok/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.6.0a0+df5bbc09d1.nv24.11
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset

In [12]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")


Loading ESB diagnostic dataset - AMI clean split...
Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples


In [13]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()


Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize VARSTok


In [3]:
# Initialize the tokenizer
print("Loading VARSTok tokenizer...")
tokenizer = get_tokenizer('varstok', device=device)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
print(f"  Token rate: {tokenizer.tokens_per_second} Hz")


Loading VARSTok tokenizer...


/users/arsaikia/benchmark-audio-tokenizer/.venv-varstok/lib/python3.12/site-packages/einops/einops.py:717: SyntaxWarning: invalid escape sequence '\s'
  """
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


there are 1 layers


/users/arsaikia/benchmark-audio-tokenizer/examples/../src/audio_tokenizers/implementations/../../repos/FunResearch/VARSTok/decoder/pretrained.py:71: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this ex

Tokenizer loaded: VARSTokWrapper(checkpoint='None', device='cuda', sample_rate=24000)
  Input sample rate: 24000 Hz
  Output sample rate: 24000 Hz
  Codebook size: 4,096
  Downsample rate: 652x
  Token rate: 36.81 Hz


## Tokenize and Reconstruct Audio Samples


In [19]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "128"

# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor and ensure correct shape (1, T) for WavTokenizer
    audio_tensor = torch.from_numpy(audio_array).float()
    if audio_tensor.dim() == 1:
        audio_tensor = audio_tensor.unsqueeze(0)  # (T,) -> (1, T)
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    print(f"Token shape: {tokens.shape}")
    print(f"Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")
    
    # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Decode back to audio
    print("\nDecoding tokens back to audio...")
    reconstructed, decode_info = tokenizer.decode(tokens)
    
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
    
    # Handle shape: (1, 1, T) or (1, T) or (T,)
    rec = reconstructed
    if rec.dim() == 3:
        rec = rec[0, 0] if rec.shape[1] == 1 else rec[0]
    elif rec.dim() == 2:
        rec = rec[0]
    rec_np = rec.detach().cpu().numpy() if torch.is_tensor(rec) else rec
    
    # Store results
    results.append({
        'original': audio_array,
        'original_sr': sr,
        'reconstructed': rec_np,
        'reconstructed_sr': decode_info['output_sample_rate'],
        'tokens': token_values,
        'transcript': transcript,
        'duration': duration
    })
    
print(f"\n{'='*60}")
print("Processing complete!")



Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Token shape: torch.Size([1, 1, 312])
Number of tokens: 312
Tokens per second: 26.5

First 20 discrete token IDs:
[ 1788 13909 13545 15249  7017 14465 10305  1313 12621  6508 12420 13133
  6209  1292 14181   842 15795  4297 14914  6483]
Token ID range: [1, 16383]
Unique tokens used: 270

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 1, 282560])
Output sample rate: 24000 Hz

Processing Sample 2 (index 1)
Duration: 11.70 seconds
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokenizing...
Token shape: torch.Size([1, 1, 271])
Number of tokens: 271
Tokens per second: 23.2

First 20 discrete token IDs:
[13082  3700 15988 15021 14846 1075

## Play Original vs Reconstructed Audio


In [20]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print(f"\nOriginal Audio ({result['original_sr']} Hz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    print(f"\nReconstructed Audio ({result['reconstructed_sr']} Hz):")
    display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 12-bit tokens (log2(4096) = 12)
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")



Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 312 (26.5 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (24000 Hz):



Compression ratio: 603.6x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 271 (23.2 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (24000 Hz):



Compression ratio: 690.8x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 298 (25.5 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (24000 Hz):



Compression ratio: 627.7x


## Summary

VARSTok successfully:
- Encodes audio to discrete tokens at a variable rate <40 Hz
- Uses a single codebook with 4,096 tokens
- Provides efficient compression for audio representation
- Supports speech, audio, and music
- Can decode tokens back to 24 kHz audio

The discrete tokens can be used for:
- Audio compression and storage
- Input to language models for audio generation
- Audio editing and manipulation tasks
- Building blocks for audio synthesis pipelines

**Key Advantages:**
- **<40 tokens/second**: Variable Frame rate
- **Single codebook**: Simple architecture, easy to use with standard language models
- **Universal support**: Works well with speech, audio, and music
- **High quality**: Trained on 585 hours of LibriTTS
